In [133]:
from dotenv import load_dotenv
load_dotenv()

True

In [134]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

In [135]:
from dataclasses import dataclass

# 실행 컨텍스트 정의 (누가 실행하는지 식별)
@dataclass
class Context:
    user_id: str
    app_name: str

In [136]:
from typing import TypedDict

# LLM이 추출해야 할 정보의 구조를 정의
class UserInfo(TypedDict):
    personal_info: str
    preference: str

In [137]:
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool, ToolRuntime
from langchain_core.tools import InjectedToolArg
from typing import Annotated

# Tool 정의: 사용자의 정보를 조회
@tool
def get_user_info(runtime: Annotated[ToolRuntime, InjectedToolArg]) -> str:
    """
    현재 사용자의 정보 조회 (시스템 내부용 도구)
    """
    user_id = runtime.context.user_id
    app = runtime.context.app_name

    print("user_id, app:",user_id, app)
    
    # 해당 네임스페이스의 모든 메모리 검색
    memories = runtime.store.search((user_id, app))

    print("memories:",memories)

    if not memories:
        return "기록된 정보 없음"

    results = []
    for item in memories:
        # 저장된 데이터 구조(UserInfo)에 맞춰 필드 확인
        data = item.value
        # 저장할 때 사용한 Key들을 확인해서 문자열로 변환
        if "personal_info" in data:
            results.append(f"- 개인정보: {data['personal_info']}")
        if "preference" in data:
            results.append(f"- 선호도: {data['preference']}")

    return "\n".join(results) if results else "데이터 형식 불일치로 읽을 수 없음"

In [138]:
import uuid

@tool
def save_user_info(user_info: UserInfo, runtime: Annotated[ToolRuntime, InjectedToolArg]):
    """
    사용자의 정보를 저장하거나 업데이트
    """
    # 1. 실행 컨텍스트에서 user_id 가져오기
    user_id = runtime.context.user_id
    app = runtime.context.app_name
    store = runtime.store

    print("store:",store)

    # 2. Store에 데이터 저장 (put(namespace, key, value))
    memory_key = str(uuid.uuid4())
    store.put((user_id, app),memory_key, user_info)

    return f"정보가 안전하게 저장되었습니다. (ID: {memory_key})"

In [139]:
system_message = "당신은 사용자의 정보를 기억하는 비서입니다. 사용자가 자신의 정보를 말하면 반드시 'save_user_info' 도구를 사용하여 저장하세요."

In [140]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[get_user_info, save_user_info], 
    store=store,
    context_schema=Context,
    system_prompt=system_message
)

In [141]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 이제 '김일남'이야"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


store: <langgraph.store.memory.InMemoryStore object at 0x000001BCE4A20910>


In [142]:
response

{'messages': [HumanMessage(content="내 이름은 이제 '김일남'이야", additional_kwargs={}, response_metadata={}, id='f5be322f-9cff-4df6-8dbe-f1f64b6e11ca'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"preference": "\\uc5c6\\uc74c", "personal_info": "\\uc774\\ub984: \\uae40\\uc77c\\ub0a8"}}'}, '__gemini_function_call_thought_signatures__': {'ceee74e3-bcc6-492d-9b52-dbddaca6b9ce': 'EtAKCs0KAXLI2ny/rW/TDaw4PC9Nfc0rJTTa6qAAnzj7y9eYLzosUr/C+N5vaat0pkEdiwxcwUdazZeOSdwg5e5Qvk+zrkGqKTEgsNqyJ6m7ZxbNtpLeF8ncO9No4p1lthWfs0HojkeRFaITPOUaYqtPPBrrvERWuLyD8VjQpVCdI1ojOPYYcivlDxoWSEjah/G6ssI0RqVT2c+jAg5IW9BaW6rsUuMRGH3JbzdnGiOISzFS+K2wmnhnaZdc6Vfx2zXO0yKDSry0s6XhRE+zbABLZ6tXk+JPTZ9+wNZbN6FJtCuUuoIEcRsuA8a0CZwuWiT7L/tvjXYqFofUKpIPEAvFmbRvzTZBlLooJYSLzpAuHpGKkn7w0XhShwlL/TDeLC7KSQoXVnAZInrSAmtUI3EZiXe4rN2kzxl7qopzKlxpaJIkGfz5YdRISlxO+T+KGei23M2PDQDnetOE+sHk3ycws8p+GK1lh+J6nBwr20dGJFjWOjlMdKaB8LB69dfrKOMu4Jkl2Udr6MzOliQMj7KDqrV5egP56NKF8L0FETgm9U

In [143]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 아이스 아메리카노를 좋아해"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


store: <langgraph.store.memory.InMemoryStore object at 0x000001BCE4A20910>


In [144]:
response

{'messages': [HumanMessage(content='나는 아이스 아메리카노를 좋아해', additional_kwargs={}, response_metadata={}, id='5a66dc21-b9f8-4eb6-abc8-7a66a2c49dab'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"personal_info": "\\uc54c\\ub824\\uc9c4 \\uc815\\ubcf4 \\uc5c6\\uc74c", "preference": "\\uc544\\uc774\\uc2a4 \\uc544\\uba54\\ub9ac\\uce74\\ub178\\ub97c \\uc88b\\uc544\\ud568"}}'}, '__gemini_function_call_thought_signatures__': {'4f19208b-aa1d-4e3f-afec-bd0d4002bfb8': 'EpwKCpkKAXLI2nxYGdwt7VgtYmoi+v66HdRap9uezW+HY+8b0gDJDAI3rW0jWjEUig6zOeRkLqq/Q2M8c+BhjNk7qHjJ6M3wNhS6nES/UMYvpvikXRgkC/15L3nUTpCcgIxJyhDaOsNexJfyWbUPDaWGuC6jab6vgwYtQjDt4NpNsfwxAyF3Ym3jwE2oWw+70ydzOtXsfxzuM3mPoaDxJTT4ca02WbjldauwLCxm6KxUdR/StLJ6rip1dmweMz+uWNJcYW9gvMc/hKfIDyL9Z9PbmB51l1YjYyY6wfiB0nNGlQgQwpZBzkhhjaea1knjjnSQQMF/bfMFzi5Ax+z4x1Y1GBdyTB+cu5C6pGD6YfeNfjhzEP8kPtyxaFDC71Z9NF50P+5wxqE6ym4uZA6yBqBZB4DLE+3q9yriXhfo2cna4BE9zrevA6sA3yoSFsDu/u8ABKsStp7A8eTfNgsvz26

In [145]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나는 랭체인 공부를 좋아해. 학습용 챗봇 프로젝트를 진행 중이야."}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


store: <langgraph.store.memory.InMemoryStore object at 0x000001BCE4A20910>


In [146]:
response

{'messages': [HumanMessage(content='나는 랭체인 공부를 좋아해. 학습용 챗봇 프로젝트를 진행 중이야.', additional_kwargs={}, response_metadata={}, id='90f9187f-5fe4-4aa8-b39b-8a1ecfdadba2'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'save_user_info', 'arguments': '{"user_info": {"personal_info": "\\ud559\\uc2b5\\uc6a9 \\ucc57\\ubd07 \\ud504\\ub85c\\uc81d\\ud2b8\\ub97c \\uc9c4\\ud589 \\uc911\\uc784", "preference": "\\ub7ad\\uccb4\\uc778(LangChain) \\uacf5\\ubd80\\ub97c \\uc88b\\uc544\\ud568"}}'}, '__gemini_function_call_thought_signatures__': {'4d095671-6c16-4d86-af7b-69496f9bfc86': 'EvYICvMIAXLI2nxNK37mYgs8QAq74YYojm4Pvf6PLdtfk3Ylap/72jh6X3EFqmGltbOjXsE/fCNBnGAr1S37DrDFkdPXasBOlZoquoAc9+9SuJbWwM+qbVju6l3RU5wnsOlVehKV/h2q03i/A9S+uycbpgBejQ8ZSMhbQ+L6RJp18yPyaDDuC+NPA09jM14Sq5FTmu5y7E3n+JwlKK+9Q5L7GrAmzQLVYCTvmS4faqlRpvTN7MtbZB2Oj2dfOHxkdXijCx12TaxiLbuTBnTLpIgKjbjcwc6KmUxniRhLkycNwpK117842vd6IvxseXEHTEL8/CTsTLV1D8e7wI0LGZiMyUPQDJyO6N5MpuFYrNgbUW4zMpVKzT0DRZ5MozGSk4qN9/YU5kV1CgT+exIixPTsH7G

In [147]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "나에 대해 알고 있는 모든 것을 말해줘"}]},
    context=Context(user_id="user_001", app_name="personal_assistant")
)

c:\Users\dandycode\Documents\GitHub\kor-it-langchain-class\ch10\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=Context(user_id='user_001...me='personal_assistant'), input_type=Context])
  return self.__pydantic_serializer__.to_python(


user_id, app: user_001 personal_assistant
memories: [Item(namespace=['user_001', 'personal_assistant'], key='aaef962d-6f75-409c-8cde-69d36d822540', value={'personal_info': '이름: 김일남', 'preference': '없음'}, created_at='2026-01-31T14:18:16.437158+00:00', updated_at='2026-01-31T14:18:16.437158+00:00', score=None), Item(namespace=['user_001', 'personal_assistant'], key='4977f42f-894c-43c5-9448-036680069597', value={'personal_info': '알려진 정보 없음', 'preference': '아이스 아메리카노를 좋아함'}, created_at='2026-01-31T14:18:55.813050+00:00', updated_at='2026-01-31T14:18:55.813050+00:00', score=None), Item(namespace=['user_001', 'personal_assistant'], key='ef7a98c9-5a0c-41cb-ba3a-32d20a41b32d', value={'personal_info': '학습용 챗봇 프로젝트를 진행 중임', 'preference': '랭체인(LangChain) 공부를 좋아함'}, created_at='2026-01-31T14:19:52.024804+00:00', updated_at='2026-01-31T14:19:52.024804+00:00', score=None)]


In [149]:
response

{'messages': [HumanMessage(content='나에 대해 알고 있는 모든 것을 말해줘', additional_kwargs={}, response_metadata={}, id='a966506b-ab07-4e48-9837-7d79d92ee779'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_user_info', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'9143dade-3432-4aa0-9fdf-6f911aacdbe8': 'EvIDCu8DAXLI2nzz+QiID8tFI6sLaK/ChJB+IMubhq0TLcXEm5Z3mXXsFSjkcOuBpAbc79GYfsOcYv7AjCA+i4TDa5EhZWXRUduM89UgBECpRE2zX5Tf8v+4yJCft4M9SGGHtL31/JFZTOTLPmICbczjSNavT8RPPfQX0W93CdIl/UjfykHzT/Bn5rWfNTMUWdUzglxnC3H745pTfGzXboREaR1sF06DEsKlsG40lnYIuorqL2WxXoXrIrcnRZtlRioeCyNnFtVhOKOS9UlAsDaO7AtzHbFX4fwF5QKz526B7bC3163VZP4H7g8AVHolTZTHWdWXiSbT/l3B7QXFyPe+7V+jihc29LC9cVeV/U/8sIWY2cDbXO+QkffwICG8WvHagyrNiC/WPxyshoLEisWqcdc/4o3SzbwlnyRth/ZohF3lS5V0+ZRK5aR3zOEoQg0ZNcHqAFBOo5uLopoy9+4yPlJl1XwyH4JX0qZdbhqYMk+hy2jkYI3d293i5SeevZpl/xGfosSPMKQLHQivHV4KthOUoZBrRRBIFMJX7R29fBPwytKbtP8BeZwy/4SiTpU1GbJRNyiSjV6So1f3eEb8+VimUjtplLr8YU/s7eZxET/NRrrHkRDyf5utY2XALb/IHhp7lE3T5URED